### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structed output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("groqAPIKey")
model = init_chat_model("groq:openai/gpt-oss-120b")

In [3]:
### Using Pydantic to make the output structured.

from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description = "The title of the movie")
    year:int = Field(description = "This year the movie was released")
    director:str = Field(description = "The director of the movie")
    rating:float = Field(description="The movies rating out of 10")
    

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020752C14680>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020752AD9670>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'yea

In [17]:
response = model_with_structure.invoke("Provide details about Inception")
response

""" Output

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

"""

" Output\n\nMovie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)\n\n"

### We can also create MESSAGE alongside parsed structure.

In [14]:
response = model.with_structured_output(Movie,include_raw=True)
response.invoke("Provide details about inception")


""" Output
{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. Likely we should call the function Movie with appropriate data. Provide director, rating, title, year. Let\'s recall: Inception directed by Christopher Nolan, rating maybe 8.8 (IMDb). Year 2010. Title "Inception". Provide that via function.', 'tool_calls': [{'id': 'fc_d100c87f-96f8-4c81-aaee-f7c0fd733f85', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 155, 'total_tokens': 272, 'completion_time': 0.252261659, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.00669129, 'prompt_tokens_details': None, 'queue_time': 0.04690497, 'total_time': 0.258952949}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2134-98a0-77f0-9f06-0a7025791d18-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'fc_d100c87f-96f8-4c81-aaee-f7c0fd733f85', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 117, 'total_tokens': 272, 'output_token_details': {'reasoning': 65}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}
"""

' Output\n{\'raw\': AIMessage(content=\'\', additional_kwargs={\'reasoning_content\': \'User wants details about the movie Inception. Likely we should call the function Movie with appropriate data. Provide director, rating, title, year. Let\'s recall: Inception directed by Christopher Nolan, rating maybe 8.8 (IMDb). Year 2010. Title "Inception". Provide that via function.\', \'tool_calls\': [{\'id\': \'fc_d100c87f-96f8-4c81-aaee-f7c0fd733f85\', \'function\': {\'arguments\': \'{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}\', \'name\': \'Movie\'}, \'type\': \'function\'}]}, response_metadata={\'token_usage\': {\'completion_tokens\': 117, \'prompt_tokens\': 155, \'total_tokens\': 272, \'completion_time\': 0.252261659, \'completion_tokens_details\': {\'reasoning_tokens\': 65}, \'prompt_time\': 0.00669129, \'prompt_tokens_details\': None, \'queue_time\': 0.04690497, \'total_time\': 0.258952949}, \'model_name\': \'openai/gpt-oss-120b\', \'system_fingerprint\':

### Nested Structure

In [24]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]   ## Nesting the Actor class
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about movie Inception")
response

""" Output

MovieDetails(title='Inception', year=2010,
cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'),
 Actor(name='Joseph Gordon-Levitt', role='Arthur'), 
 Actor(name='Elliot Page', role='Ariadne'),
  Actor(name='Tom Hardy', role='Eames'),
   Actor(name='Ken Watanabe', role='Saito'),
    Actor(name='Cillian Murphy', role='Robert Fischer'),
 genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)
 """

" Output\n\nMovieDetails(title='Inception', year=2010,\ncast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'),\n Actor(name='Joseph Gordon-Levitt', role='Arthur'), \n Actor(name='Elliot Page', role='Ariadne'),\n  Actor(name='Tom Hardy', role='Eames'),\n   Actor(name='Ken Watanabe', role='Saito'),\n    Actor(name='Cillian Murphy', role='Robert Fischer'),\n genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)\n "

### TypeDict

TypedDict provides a simpler alternative using python's built-in typing, ideal when you don't need
runtime validation.

In [32]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """ A movie with Details """
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[str,..., "The year of the movie"]
    director: Annotated[str,...,"The director of the movie"]

model_with_typeDict_structure = model.with_structured_output(MovieDict)
response = model_with_typeDict_structure.invoke("Please provide the details of the movie avengers")
response


""" Output

{'director': 'Joss Whedon', 'title': 'The Avengers', 'year': '2012'}

"""

" Output\n\n{'director': 'Joss Whedon', 'title': 'The Avengers', 'year': '2012'}\n\n"

### Nested Structure with TypeDict

In [ ]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str


class MovieDict(TypedDict):
    """ A movie with Details """
    title: Annotated[str, ..., "The title of the movie"]
    cast: list[Actor]
    genres: list[str]
    year: Annotated[str,..., "The year of the movie"]
    director: Annotated[str,...,"The director of the movie"]



model_with_typeDict_structure = model.with_structured_output(MovieDict)
response = model_with_typeDict_structure.invoke("Please provide the details of the movie avengers")
response



# Output
"""
{'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'}],
 'director': 'Joss Whedon',
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': '2012'}
"""

{'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'}],
 'director': 'Joss Whedon',
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': '2012'}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.